# A2.6 · Ingress: marking untrusted content at the door

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.5 · The non-human identity lifecycle](https://spbreed.github.io/cyber-commons/lessons/A2.5.html)**.

| | |
|---|---|
| Tools used | LLM Guard, agentgateway, Llama Guard 4, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Tag every span at ingress, then show the same payload refused through six different entry paths.

**Why a security engineer needs it.** Concatenation destroys the one fact that separates an operator instruction from an attacker's: where it came from. The control it builds is: provenance tagging at every ingress point, and a rule that only trusted origins may select a tool.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

By the time the model sees it, the context window is one flat string. The operator's instruction, the user's question and a paragraph from a stranger's web page are indistinguishable — unless something attached an origin to each span before they were concatenated.

> **At CyberTravels.** A hotel description, a booking note and CyberTravels' operator prompt arrive at the model as one flat string. Marking each span with where it came from is what makes “a retrieved document may not select a tool” expressible at all. R3.

## 2 · The framework

```
   before                          after
   +-------------------------+     +---------------------------+
   | system prompt           |     | [principal] system prompt |
   | user question           |     | [principal] user question |
   | retrieved document      |     | [data]      retrieved doc |
   | tool result             |     | [data]      tool result   |
   +-------------------------+     +---------------------------+
   one flat string                 origin travels with the span

   rule that becomes possible: a [data] span may not select a tool
```

**Mitigates: T6 Intent Breaking, direct and indirect · T1 Memory Poisoning · T12 Communication Poisoning.**

This is the control for the largest risk in Chapter 1.

A1.3 worked because the context window is one flat string. Everything —
operator instruction, user question, retrieved document, tool result, peer
message — arrives as tokens with no marker for where it came from. The
distinction the operator believed in is destroyed by the concatenation.

The control is to **stop flattening**: attach an origin to every span *before*
assembly, carry it everywhere the span goes, and make one rule out of it.

> **Only spans from a trusted origin may select a tool.**

Three properties do the work:

**Tag at every ingress point.** Not just retrieval — tool results, MCP tool
descriptions, memory reads, and inter-agent messages are all ingress. A path you
did not tag is a path with no control on it.

**Carry the tag into memory.** This is what stops A1.4. A summary written from a
trust-0 document is itself trust-0, and if the tag is dropped on write the
poison becomes a fact.

**Let untrusted content still be useful.** The document is read, summarised,
quoted and reasoned about. What it may not do is choose an action. Refusing to
*read* untrusted content would refuse the entire use case.

Note what this does not do: it does not detect malicious text. It never looks at
the content at all, which is exactly why rephrasing does not defeat it.

> **What this control closes.**
>
> The largest control in the chapter. It never inspects content, so rewriting the payload does not help — the check is on **origin**, which the attacker cannot change.

## Your turn

List every place text enters your agent's context and check which of them attaches an origin. The untagged ones are the paths where this control does not exist, whatever the design document says.

---

**Next → [A2.7 · Attribution: an audit trail that answers "who"](https://spbreed.github.io/cyber-commons/lessons/A2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*